# **Forecasting Pipeline**

This pipeline automates **time series forecasting** across different datasets using **deep learning models**. It supports **incremental learning, fine-tuning, and multiprocessing** to handle multiple series efficiently.

---

## **Pipeline Overview**

### **1. Data Handling**
- Supports **Volve (oil production), UNISIM (reservoir simulation), and OPSD (energy generation)**.
- Loads, cleans, and prepares time series data.
- Organizes wells by dataset size for efficient processing.

### **2. Model Training & Fine-Tuning**
- **First Iteration**: Trains base models from scratch.
- **Subsequent Iterations**: Fine-tunes models with new data chunks.
- Uses **sliding window forecasting** to adapt over time.

### **3. Prediction & Evaluation**
- Generates **step-by-step forecasts** using trained models.
- Applies **optional filters** (e.g., Kalman filter) to refine predictions.
- Computes **MAE & MSE** for model evaluation.
- Plots results and aggregates performance metrics.

---

## **Key Functions**

### **🔹 `train_and_evaluate_generic()`**
**Orchestrates the full forecasting process**:
- Loads data, initializes training, and runs fine-tuning loops.
- Uses multiprocessing for efficient parallel execution.
- Supports **custom data loaders, variable mappings, and post-processing filters**.

### **🔹 `train_base_models()`**
Trains initial models for each active well and saves them for fine-tuning.

### **🔹 `prepare_args_for_fine_tuning()`**
Prepares data and arguments for model fine-tuning across multiple wells.

### **🔹 `apply_filter_to_predictions()`**
Applies **post-processing filters** to smooth predictions.

### **🔹 `evaluate_and_plot_all_wells()`**
Computes and visualizes **model performance** across datasets.

---

## **Dataset Summary**

| Dataset  | Description                | Target Variable                   | Frequency |
|----------|----------------------------|-----------------------------------|------------|
| **Volve**  | Offshore oil production  | `BORE_OIL_VOL`                   | Daily      |
| **UNISIM** | Reservoir simulation     | `QOOB`                            | Daily      |
| **OPSD**   | Energy generation (wind) | `GB_GBN_wind_generation_actual`  | 30 min     |

---

## **Why Use This Pipeline?**
✅ **Automated Forecasting & Fine-Tuning**  
✅ **Parallel Processing for Faster Training**  
✅ **Scalable for Different Datasets & Models**  
✅ **Adaptive Learning with Sliding Window**  
✅ **Customizable Preprocessing & Post-Processing**  

In [ ]:
%matplotlib widget
# %%capture capturado
# %%capture --no-stderr
# ──────────────────────────────────────────────────────────────────────────────
# Stdlib
# ──────────────────────────────────────────────────────────────────────────────
import multiprocessing as mp
import os
from pathlib import Path
from typing import Any, Callable, Dict, List, Sequence
import warnings
import logging

"""Suppress TensorFlow and addon warnings for a cleaner console."""
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOG_LEVEL'] = '3'
warnings.filterwarnings(
    'ignore',
    category=UserWarning,
    module='tensorflow_addons'
)
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

import os
import time
import multiprocessing
from typing import Any, List, Dict, Callable, Union

import numpy as np
import pandas as pd

# Imports dos módulos de preparação, avaliação e treinamento
from data.data_preparation import (
    filter_data_for_iteration,
    prepare_train_test_sets,
    initialize_prediction_lists,
    calculate_total_iterations,
    organize_wells_by_df_size,
    apply_custom_kalman_filter
)
from evaluation.evaluation import (
    evaluate_and_plot_all_wells,
    evaluate_and_plot_if_needed,
    compute_metrics,
    plot_results,
    evaluate
)
from training.train_utils import prepare_args_for_fine_tuning, fine_tune_and_predict_well
from utils.utilities import (
    delete_all_files_in_folder,
    apply_filter_to_predictions,
    print_style
)
from training.models_forecast import train_and_evaluate_disruptive

# Importa a classe DataSource do novo módulo unificado de carregamento
from data.data_loading import DataSource

In [ ]:
from pathlib import Path

def _suffix_path(base: str | Path, suffix: str) -> Path:
    p = Path(base)
    return p.with_stem(f"{p.stem}_{suffix}")


def _downsample(df: pd.DataFrame | Dict[str, pd.DataFrame], step: int, wells: Sequence[str]):
    if isinstance(df, dict):
        return [df[w].iloc[::step] for w in wells]
    return df.iloc[::step]


# ──────────────────────────────────────────────────────────────────────────────
# Core pipeline class
# ──────────────────────────────────────────────────────────────────────────────
class WellForecastPipeline:  # noqa: D101
    def __init__(
        self,
        dataset: str,
        wells: List[str],
        serie_name: str,
        data_path: str,
        forecast_steps: int,
        window_size: int,
        train_windows: int,
        fine_tuning_windows: int,
        model_type: str,
        architecture_name: str = "Generic",
        sample_time: int = 1,
        model_path: str | Path = "best_model.keras",
        cum_sum: bool = False,
        data_loader_kwargs: Dict[str, Any] | None = None,
        variable_mapping: Dict[str, str] | None = None,
        filter_postprocess: Callable | None = None,
    ) -> None:
        # (unchanged – omitted for brevity)
        self.dataset = dataset
        self.wells = wells
        self.serie_name = (
            variable_mapping.get(serie_name) if variable_mapping else serie_name
        )
        self.data_path = data_path
        self.forecast_steps = forecast_steps
        self.window_size = window_size
        self.train_windows = train_windows
        self.fine_tuning_windows = fine_tuning_windows
        self.model_type = model_type
        self.architecture_name = architecture_name
        self.sample_time = sample_time
        self.model_path = Path(model_path)
        self.cum_sum = cum_sum
        self.data_loader_kwargs = data_loader_kwargs or {}
        self.variable_mapping = variable_mapping
        self.filter_fn = filter_postprocess or apply_custom_kalman_filter

        # runtime state
        self.df_list: List[pd.DataFrame] = []
        self.active_wells: List[int] = []
        self.y_test_list: List[List[float]] = []
        self.y_pred_list: List[List[float]] = []
        self.total_iters: int = 0

    # Public API ────────────────────────────────────────────────────────────
    def run(self):  # noqa: D401
        self._load_data()

        # ★ Re‑use multiprocessing pool for the whole run
        with mp.Pool(processes=min(len(self.wells), os.cpu_count() or 1)) as pool:
            self.pool = pool  # keep reference
            self._iterate()

        self._final_eval()

    # Data loading (unchanged except IO cache) ────────────────────────────
    def _load_data(self):  # noqa: D401
        cfg = {
            "name": self.dataset,
            "wells": self.wells,
            "serie_name": self.serie_name,
            "load_params": {
                **self.data_loader_kwargs,
                "data_path": self.data_path,
                "cum_sum": self.cum_sum,
            },
            "variable_mapping": self.variable_mapping,
            "features": [
                "BORE_GAS_VOL",
                "CE",
                "delta_P",
                "PI",
                "AVG_DOWNHOLE_PRESSURE",
                "BORE_WAT_VOL",
                "ON_STREAM_HRS",
                "Tempo_Inicio_Prod",
                "Taxa_Declinio",
                "BORE_OIL_VOL",
            ],
        }
        raw = DataSource(cfg).get_loader().load()
        self.df_list = _downsample(raw, self.sample_time, self.wells)
        self.wells, self.df_list = organize_wells_by_df_size(
            self.wells, self.df_list
        )
        self.active_wells = list(range(len(self.wells)))
        self.y_test_list, self.y_pred_list = initialize_prediction_lists(
            len(self.wells)
        )
        self.total_iters = calculate_total_iterations(self.df_list)
        print(f"[Data] {len(self.wells)} wells · {self.total_iters} iterations")

    # Iteration loop – uses shared pool ───────────────────────────────────
    def _iterate(self):  # noqa: D401
        for it in range(self.total_iters):
            if it and it % 1000 == 0:
                print(f"[Iter] Fine‑tuning {it + 1}/{self.total_iters}")
                self._final_eval()

            sets = filter_data_for_iteration(
                self.df_list,
                self.window_size,
                self.serie_name,
                self.forecast_steps,
                it,
                self.active_wells,
                self.train_windows,
                self.fine_tuning_windows,
                cum_sum=self.cum_sum,
            )
            if not sets:
                continue
            self._handle_iteration(sets, it)

    def _handle_iteration(self, sets, it: int):  # noqa: D401
        # (unchanged up to pool.map)
        X_tr, y_tr, X_ts, y_ts, max_tr, scalers = prepare_train_test_sets(
            sets, self.model_type, well_idx=0
        )

        if self.model_type == "DL" and it == 0:
            self._train_base_models(sets)
            return

        args, self.active_wells = prepare_args_for_fine_tuning(
            sets,
            X_ts,
            y_ts,
            max_tr,
            scalers,
            self.model_type,
            str(self.model_path),
            self.wells,
            self.active_wells,
            self.cum_sum,
            it,
        )
        if not args:
            return

        # ★ use already‑allocated pool
        for result in self.pool.map(fine_tune_and_predict_well, args):
            if not result:
                continue
            idx, y_true, y_pred = result
            self.y_test_list[idx].extend(y_true)
            self.y_pred_list[idx].extend(y_pred)

    # Base training – auto batch size ─────────────────────────────────────
    def _train_base_models(self, sets):  # noqa: D401
        import GPUtil  # lazy import – only if GPU exists

        try:
            free_mem = max(g.memoryFree for g in GPUtil.getGPUs())
            # Rough rule‑of‑thumb: 1 MB / sample
            auto_bs = max(16, min(64, free_mem // 1024))
        except Exception:
            auto_bs = 32

        for i in self.active_wells:
            X_b, y_b, *_ = prepare_train_test_sets(sets, self.model_type, well_idx=i)
            path = _suffix_path(self.model_path, self.wells[i].replace("/", "_"))
            train_and_evaluate_disruptive(
                X_b,
                y_b,
                model_path=str(path),
                fine_tune=False,
                architecture_name=self.architecture_name,
                batch_size_override=auto_bs,  # ★
            )

    # Evaluation
    def _final_eval(self):
        y_pred_filt = apply_filter_to_predictions(self.y_pred_list, self.filter_fn)
        for tag, preds in {"Kalman": y_pred_filt, "No Filter": self.y_pred_list}.items():
            print_style(tag)
            evaluate_and_plot_all_wells(
                self.dataset,
                self.wells,
                self.y_test_list,
                preds,
                self.window_size,
                self.forecast_steps,
                metrics_accumulator=[],
                method=tag,
            )


# ──────────────────────────────────────────────────────────────────────────────
# Public helpers
# ──────────────────────────────────────────────────────────────────────────────

def run_pipeline(**kwargs):
    """Instantiate and run a `WellForecastPipeline`."""
    WellForecastPipeline(**kwargs).run()


def train_and_evaluate_generic(
    forecast_steps: int,
    window_size: int,
    dataset: str,
    wells: List[str],
    serie_name: str,
    data_path: str,
    model_type: str,
    sample_time: int,
    train_windows: int,
    fine_tuning_windows: int,
    model_path: str,
    cum_sum: bool,
    data_loader_kwargs: Dict[str, Any] | None = None,
    variable_mapping: Dict[str, str] | None = None,
    filter_postprocess: Callable | None = None,
    architecture_name: str = "Generic",
) -> None:
    run_pipeline(
        dataset=dataset,
        wells=wells,
        serie_name=serie_name,
        data_path=data_path,
        forecast_steps=forecast_steps,
        window_size=window_size,
        model_type=model_type,
        sample_time=sample_time,
        train_windows=train_windows,
        fine_tuning_windows=fine_tuning_windows,
        model_path=model_path,
        cum_sum=cum_sum,
        data_loader_kwargs=data_loader_kwargs,
        variable_mapping=variable_mapping,
        filter_postprocess=filter_postprocess,
        architecture_name=architecture_name,
    )

In [ ]:
import os
import multiprocessing
from common.config_wells import DATA_SOURCES
from utils.utilities import delete_all_files_in_folder

# ─────────────────────────────────────────────────────────────────────────────
# Global – set forkserver + XLA
# ─────────────────────────────────────────────────────────────────────────────

try:
    mp.set_start_method("forkserver", force=True)
except RuntimeError:  # already set inside interactive env
    pass

# Enable XLA JIT (TensorFlow ≥ 2.6)
try:
    tf.config.optimizer.set_jit(True)
except Exception:  # pragma: no cover – CPU‑only or older TF
    pass

if __name__ == '__main__':
    # multiprocessing.set_start_method('spawn', force=True)
    
    # Exemplo: filtrar apenas os DATA_SOURCES desejados
    selected_names = ["VOLVE", "UNISIM", "OPSD"]
    # Para selecionar apenas "VOLVE", descomente a linha abaixo:
    selected_names = ["UNISIM"]
    filtered_data_sources = [data_source for data_source in DATA_SOURCES if data_source["name"] in selected_names]
    for _ in range(1):
        for data_source in filtered_data_sources:
            print(f"\nExecuting data_source: {data_source['name']}")
            # delete_all_files_in_folder(os.path.dirname(data_source['model_path']))
            train_and_evaluate_generic(
                forecast_steps=56,
                window_size=7,
                dataset=data_source['name'],
                wells=['Prod-10'],
                serie_name=data_source['load_params']["serie_name"],
                data_path=data_source['load_params']['data_path'],
                model_type="DL",
                sample_time=1,
                train_windows=150,
                fine_tuning_windows=150,
                model_path=data_source["model_path"],
                cum_sum=True,
                data_loader_kwargs=data_source.get("load_params"),
                variable_mapping=data_source.get("variable_mapping"),
                filter_postprocess=data_source.get("filter_postprocess"),
                #architecture_name: N_day, Golem, Generic, Encoder - Check models for more details
                architecture_name='Generic'
            )